# ESM3 alpha dial — prediction

What the dial does to predictive performance on the ESM3 sweep, against the boosting
base on the same folds.

**This notebook draws and does not aggregate** — see the geometry notebook's header for
why that rule is load-bearing. In short: `alpha_grid` averages model seeds inside each
fold and takes the interval over folds, and no figure here recomputes a mean.

Two figures carry the argument. The first shows two absolute levels; the second shows
their **paired difference within each fold**, which is the comparison the paper makes
and the only one whose interval means what it appears to mean.

**One rule of colour on this page: blue is test, orange is validation.** Both splits
are drawn on the same axes, because the question a reader has when they see a dial is
whether the two agree -- and two figures on two pages answer it badly. Validation is
here to be looked at, not to be chosen from: the dial is argued about in
`alpha_rank_dial.ipynb`, and no argmax is taken anywhere below.

In [ ]:
import pathlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks" / "article_figures"))

import figkit as fk
from scripts.analysis import alpha_grid as ag

fk.use_house_style()

# Validation is drawn with its own dash AND its own marker, not colour alone:
# these panels are read in print and in greyscale at least as often as on screen.
VAL_DASH = (0, (3, 1.2))
print("repo:", ROOT)

## Knobs

In [ ]:
# ================================== KNOBS ====================================
ROOT_DIR   = "results/graph/v13_esm3"   # the ESM3 sweep
NODES      = "nodedial"                 # "nodedial" (v9) | "onehot" (v8 gate) | None
MOL_SOURCE = "chemberta"
DATASET    = None          # None -> every dataset on disk; or ["cc", "hc"] / "m2or"
REGIME     = None          # None -> both; or "transductive" / "inductive"
SEED       = None          # model seed; None -> every one present
SHOW_VAL   = True          # overlay the validation curves on the same axes
COMBO      = "cls+mol"     # which head the graph rows come from

CI_LEVEL   = 0.95          # Student-t over FOLDS (seeds are averaged inside each fold)
METRICS    = None          # None -> the metric of record per dataset; or a list
LEVELS     = ("boost_full",)  # horizontal references. `naive` (the train mean) is
                              # deliberately not among them: it sits far below every
                              # arm, so it only stretches the y axis and flattens the
                              # differences the figure is about
RESOLUTION_METRIC = None   # None -> the metric of record of the first series

SAVE_FIGS  = False
FIG_DIR    = "results/article_figures/esm3"
# =============================================================================

_kw = dict(root=ROOT_DIR, mol_source=MOL_SOURCE, nodes=NODES, dataset=DATASET,
           regime=REGIME, seed=SEED, combo=COMBO)
df = ag.load(split="test", **_kw)

# The validation frame, in its own variable and never concatenated with the test one.
# It is what alpha was CHOSEN on -- the sweep writes `val_metrics_*.csv` beside its own
# metrics, and older grids get them from `val_rescore.py`. A run without them is not an
# error here; section 3 then says what to run instead of plotting a choice that was
# never made on these rows.
try:
    dfv = ag.load(split="val", **_kw)
except SystemExit as e:
    dfv = None
    print("no validation rows:", e)

SERIES = sorted(df.series.unique())
ALPHAS = sorted(df.loc[df.arm == "gate", "alpha"].dropna().unique())

In [ ]:
# ---------------------------------------------------------------- what is on disk
# The two dials must never share an axis. On a v8 GATE run alpha mixes the graph's
# OUTPUT against a frozen ESM branch; on a v9 NODE dial it mixes the graph's INPUT,
# and it runs the other way (alpha=0 is receptor identity, alpha=1 the legacy graph).
# A frame holding both puts opposite meanings on one x axis, so refuse it here rather
# than discover it in a plot.
dials = sorted(df["nodes"].astype(str).unique())
if len(dials) != 1:
    raise SystemExit(
        f"this frame holds {dials} -- two dials cannot share an axis. "
        f"Set NODES to one of them.")

cov = ag.coverage(df)
print(f"{len(SERIES)} series | {len(ALPHAS)} alphas | dial: {dials[0]}")
print("alphas:", ", ".join(f"{a:g}" for a in ALPHAS))
pin = ", ".join(f"{k}={v}" for k, v in df.attrs["pinned"].items()) or "-"
print("pinned:", pin)
cov

In [ ]:
MOF = ag.metric_for(df)                     # metric of record per dataset
DS_OF = dict(zip(df.series, df.dataset))
if RESOLUTION_METRIC is None:
    RESOLUTION_METRIC = MOF[DS_OF[SERIES[0]]]
print("metric of record:", MOF)
print("resolution computed on:", RESOLUTION_METRIC)

In [ ]:
# ------------------------------------------------- how small a difference is visible
# The floor: the model noise LEFT IN A FOLD MEAN after averaging the seeds. Every curve
# on this page is made of fold means, so a wiggle below this number is not a finding.
#
# This is also why the curves carry a CI band over folds and an I-mark for the floor,
# and never whiskers over folds: the between-fold spread is the larger quantity by a
# wide margin, and drawing it on the curve answers "how different are the folds" on a
# figure that is asking "did moving the dial change anything".
ns = ag.noise_split(df, RESOLUTION_METRIC)
FLOOR = {}
if len(ns):
    g = ns.groupby("series")[["within", "between", "floor"]].mean()
    FLOOR = g["floor"].to_dict()
    display(g.round(4))
    print("\nratio between/floor -- how much of the spread is the folds, not the model:")
    display((g["between"] / g["floor"]).round(1).rename("x"))
else:
    print(f"{RESOLUTION_METRIC!r} not in this sweep; no floor computed")

---
## 2. Performance along the dial

In [ ]:
# 2.1  THE METRIC OF RECORD ALONG THE DIAL, both splits on one axis.
# Blue is test, orange is validation. The dashed grey level is the boosting base on
# TEST; its validation twin is the lighter dashed line, so a panel where the two bases
# sit far apart is saying something about the splits, not about the dial.
# The band is the CI over folds; the grey I-mark on the left is the resolution floor
# from the cell above -- a bump shorter than that mark is model noise.
fig, ax = fk.panels(len(SERIES), ncols=min(3, len(SERIES)), w=3.3, h=2.5)

for i, s in enumerate(SERIES):
    a = ax[i]
    metric = MOF[DS_OF[s]]
    sub = df[df.series == s]
    cur = ag.curve(sub, metric, by=("series",), level=CI_LEVEL)
    drawn = fk.curve_with_band(a, cur, fk.C["gate"])
    lv = ag.levels(sub, metric, arms=LEVELS, by=("series",), level=CI_LEVEL)
    for _, r in lv.iterrows():
        fk.reference_line(a, r["mean"], r["arm"])
    if SHOW_VAL and dfv is not None:
        vsub = dfv[dfv.series == s]
        vcur = ag.curve(vsub, metric, by=("series",), level=CI_LEVEL)
        if len(vcur):
            vcur = vcur.sort_values("alpha")
            fk.band(a, vcur["alpha"], vcur["lo"], vcur["hi"], fk.C["fun"], 0.12)
            a.plot(vcur["alpha"], vcur["mean"], color=fk.C["fun"], marker="s",
                   markersize=3.4, ls=VAL_DASH)
            drawn = True
        vlv = ag.levels(vsub, metric, arms=LEVELS, by=("series",), level=CI_LEVEL)
        for _, r in vlv.iterrows():
            a.axhline(r["mean"], color=fk.C["fun"], lw=1.0, ls=fk.DASH["boost_full"],
                      alpha=0.55)
    fk.alpha_axis(a, ALPHAS)
    if not drawn:
        fk.note_empty(a)
    else:
        fk.resolution_mark(a, FLOOR.get(s))
    a.set_title(s, fontsize=8.5)
    a.set_xlabel(r"$\alpha$")
    a.set_ylabel(metric)

fig.suptitle("The metric of record along the dial", y=1.005, fontsize=11)
handles = [
    Line2D([], [], color=fk.C["gate"], marker="o", label="our graph, test"),
    Line2D([], [], color=fk.C["boost_full"], ls=fk.DASH["boost_full"],
           label="boosting base, test"),
]
if SHOW_VAL and dfv is not None:
    handles += [
        Line2D([], [], color=fk.C["fun"], marker="s", ls=VAL_DASH,
               label="our graph, validation"),
        Line2D([], [], color=fk.C["fun"], ls=fk.DASH["boost_full"], alpha=0.55,
               label="boosting base, validation"),
    ]
handles.append(Line2D([], [], color=fk.MUTED, lw=6, alpha=0.2,
                      label=f"band = {CI_LEVEL:.0%} CI over folds"))
fk.figlegend(fig, handles, ncol=3, pad_in=0.95)
fk.savefig(fig, "prediction_curves", FIG_DIR, SAVE_FIGS)
plt.show()


### 2.2 The advantage over the base, paired

In [ ]:
# 2.2  THE ADVANTAGE OVER THE BASE, PAIRED WITHIN EACH FOLD, on both splits.
# The curves above show absolute levels; this shows their difference computed fold by
# fold against that split's OWN base, which is the comparison the paper makes. Zero is
# the base. A band that clears zero along a stretch of the dial is the claim; a band
# straddling it is not -- and a claim that only clears zero on one of the two splits is
# worth knowing before it is written down.
fig, ax = fk.panels(len(SERIES), ncols=min(3, len(SERIES)), w=3.3, h=2.5)
for i, s in enumerate(SERIES):
    a = ax[i]
    metric = MOF[DS_OF[s]]
    a.axhline(0.0, color=fk.C["boost_full"], lw=1.3, ls=fk.DASH["boost_full"])
    d = ag.delta_vs(df[df.series == s], metric, ref_arm="boost_full",
                    by=("series",), level=CI_LEVEL)
    drawn = fk.curve_with_band(a, d, fk.C["gate"])
    if SHOW_VAL and dfv is not None:
        dv = ag.delta_vs(dfv[dfv.series == s], metric, ref_arm="boost_full",
                         by=("series",), level=CI_LEVEL)
        if len(dv):
            dv = dv.sort_values("alpha")
            fk.band(a, dv["alpha"], dv["lo"], dv["hi"], fk.C["fun"], 0.12)
            a.plot(dv["alpha"], dv["mean"], color=fk.C["fun"], marker="s",
                   markersize=3.4, ls=VAL_DASH)
            drawn = True
    fk.alpha_axis(a, ALPHAS)
    if not drawn:
        fk.note_empty(a)
    a.set_title(s, fontsize=8.5)
    a.set_xlabel(r"$\alpha$")
    a.set_ylabel(f"{metric} - base")

fig.suptitle("Paired advantage over the boosting base, fold by fold", y=1.005,
             fontsize=11)
dh = [Line2D([], [], color=fk.C["gate"], marker="o", label="test"),
      Line2D([], [], color=fk.C["boost_full"], ls=fk.DASH["boost_full"],
             label="the base (zero)")]
if SHOW_VAL and dfv is not None:
    dh.insert(1, Line2D([], [], color=fk.C["fun"], marker="s", ls=VAL_DASH,
                        label="validation"))
dh.append(Line2D([], [], color=fk.MUTED, lw=6, alpha=0.2,
                 label=f"band = {CI_LEVEL:.0%} CI over folds"))
fk.figlegend(fig, dh, ncol=4)
fk.savefig(fig, "prediction_delta", FIG_DIR, SAVE_FIGS)
plt.show()


### 2.3 The whole battery

In [ ]:
# 2.3  THE WHOLE BATTERY, not just the metric of record. A gain that shows up in one
# metric and nowhere else is a property of that metric.
metrics = METRICS or ag.metrics_available(df, which="headline")
metrics = [m for m in metrics if m in df.columns]
print("drawing:", metrics)

for s in SERIES:
    sub = df[df.series == s]
    have = [m for m in metrics if pd.to_numeric(sub[m], errors="coerce").notna().any()]
    if not have:
        continue
    fig, ax = fk.panels(len(have), ncols=min(4, len(have)), w=2.7, h=2.1, sharex=True)
    for j, m in enumerate(have):
        a = ax[j]
        fk.curve_with_band(a, ag.curve(sub, m, by=("series",), level=CI_LEVEL),
                           fk.C["gate"])
        if SHOW_VAL and dfv is not None:
            vc = ag.curve(dfv[dfv.series == s], m, by=("series",), level=CI_LEVEL)
            if len(vc):
                vc = vc.sort_values("alpha")
                a.plot(vc["alpha"], vc["mean"], color=fk.C["fun"], marker="s",
                       markersize=3.0, ls=VAL_DASH)
        lv = ag.levels(sub, m, arms=LEVELS, by=("series",), level=CI_LEVEL)
        for _, r in lv.iterrows():
            fk.reference_line(a, r["mean"], r["arm"])
        fk.alpha_axis(a, ALPHAS)
        a.set_title(m, fontsize=8.5)
        a.set_xlabel(r"$\alpha$")
    fig.suptitle(s, y=1.01, fontsize=10.5)
    bh = [Line2D([], [], color=fk.C["gate"], marker="o", label="our graph, test"),
          Line2D([], [], color=fk.C["boost_full"], ls=fk.DASH["boost_full"],
                 label="base, test")]
    if SHOW_VAL and dfv is not None:
        bh.insert(1, Line2D([], [], color=fk.C["fun"], marker="s", ls=VAL_DASH,
                            label="our graph, validation"))
    fk.figlegend(fig, bh, ncol=3)
    fk.savefig(fig, f"prediction_battery_{s.replace(' / ', '_').replace(' ', '')}",
               FIG_DIR, SAVE_FIGS)
    plt.show()

---
## 3. Where alpha is chosen

Not here. Both splits are on the panels above so that their agreement is visible, but
no argmax is taken on this page: reading the best alpha off a curve and then reporting
that same curve picks the number and its defence from the same rows.

The argument lives in **`alpha_rank_dial.ipynb`**, next door: every alpha ranked
*within* each (dataset, regime, molecule source) cell on validation, the mean place
plotted against the dial for both boosting heads, a line fitted through it, and that
line's slope tested against zero. The question it settles is not which alpha wins --- an
argmax over a flat surface is noise --- but whether the surface is flat at all, which is
what licenses reporting the end of the scale.

The table that used to do this (`scripts/article_tables/06_alpha_choice_not_used.py`)
is no longer part of the paper. It still runs; it is not in the runbook.

A grid written before Sep 2026 has no validation rows; `scripts/analysis/val_rescore.py
--root <this root>` adds them without retraining anything.


---
## Before quoting any alpha from this page

The orange curve on these panels is the surface a choice would be made on, and it is
drawn beside the blue one precisely so that the two can be compared by eye. That is
not the same as choosing by eye. Every reported alpha is argued for in
`alpha_rank_dial.ipynb`, which ranks on validation and tests the dial's slope against
zero -- see section 3.